# Demo: one event, every source

Montandon holds one disaster in many sources, and each records something different: USGS the magnitude, EM-DAT the losses, IDMC the displacement, CEMS the maps. No shared id joins them. This notebook takes the event you selected, searches the other sources around its time and place, and asks **Jev**, an AI decision model, which records describe the same disaster.

Then it puts everything those sources say into **one fact sheet, one chart and one map**: where they add up, and where they contradict each other. The scores are AI estimates, not facts: read *Read this before trusting it* at the end.


In [ ]:
# --- Reserved parameters (injected by the Notebook Factory; defaults keep the notebook runnable standalone) ---
run_id = ""
inputs_dir = ""
output_dir = "outputs"
area = {}                      # {id, code, name, level, country, country_name, bbox}
area_geojson = ""              # FeatureCollection with the target area
subdivisions_geojson = ""      # FeatureCollection with the aggregation subdivisions
stac_item = {}                 # the selected Montandon event (dict)
stac_item_json = ""
stac_collection = ""
stac_api_url = ""              # root of the item's catalog (Montandon)
stac_catalog_profile = ""      # "generic" | "montandon"
filters = {}                   # the run's filter tags
hazard = ""

# --- User parameters ---
confidence_threshold = 0.6     # a candidate joins the event at this probability or more (possible: 0.3 up to it)
max_candidates = 60            # most candidates to judge, the most plausible by time and distance
hazard_windows = """EQ 7 300
TS 7 1500
VO 30 200
LS 7 100
AV 7 100
FL 30 500
FF 14 300
TC 21 2500
ST 14 800
SS 14 800
DR 180 2000
WF 30 300
HT 30 1500
CW 30 1500
EP 90 3000"""                  # one hazard per line: GLIDE code, days either side of the event, radius in km
default_window_days = 14       # window for a hazard that is not in the table
default_radius_km = 500        # radius for a hazard that is not in the table
coarse_radius_km = 1500        # at least this radius when a record is only located at country level
min_cross_hazard_magnitude = 5.0   # a smaller earthquake (no tsunami flag) is only compared with earthquakes


## 1. The selected event


In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from manywidgets import Chart, Column, Legend, RangeSlider, Row, Stat

sys.path.insert(0, str(Path.cwd()))  # the helper module ships next to the notebook
import matching as M

Path(output_dir).mkdir(parents=True, exist_ok=True)
api_url = stac_api_url or os.environ.get("STAC_API_URL", "")
token = os.environ.get("STAC_API_KEY", "")
if not stac_item or not api_url:
    raise RuntimeError("Select a Montandon event when starting the run: it is the event to match.")

seed = M.record_from_item(stac_item)
rules = M.Rules.from_params(
    hazard_windows, default_window_days, default_radius_km, coarse_radius_km, min_cross_hazard_magnitude
)
days, radius = rules.window(seed)

Row(
    Stat(label="Source", value=M.SOURCE_LABELS.get(seed.source, seed.source)),
    Stat(label="Event", value=seed.title),
    Stat(label="Date", value=seed.start.date().isoformat() if seed.start else "unknown"),
    Stat(label="Countries", value=", ".join(seed.countries) or "unknown"),
    gap="16px",
)


## 2. Find the same disaster in the other sources

The search looks in every other source for records inside the hazard's time window and radius (both are parameters), then asks Jev, for each candidate, how likely it is the same disaster as the selected event. Candidates at or above the confidence threshold **join**; those between 0.3 and the threshold are listed as **possible**.


In [ ]:
candidates, searched = M.find_candidates(api_url, token, seed, max_candidates, rules)
exact = M.exact_links(seed, candidates)  # records Montandon links to the event: no need to ask
api_key = os.environ.get("OPENROUTER_API_KEY", "")
jev = M.ask_jev(seed, [c for c in candidates if c.id not in exact], api_key, rules=rules)
matches = M.match_table(candidates, exact, jev.scores)

if not api_key:
    print("No OPENROUTER_API_KEY: the candidates are not scored, and only exact links join the event.")
for error in jev.errors:
    print("A Jev call failed, its candidates have no score:", error)

joined = matches[matches.p_same >= confidence_threshold]
possible = matches[(matches.p_same >= 0.3) & (matches.p_same < confidence_threshold)]
display(Row(
    Stat(label="Candidates searched", value=searched, unit=f"within {days} d / {radius} km"),
    Stat(label="Joined", value=len(joined)),
    Stat(label=f"Possible (0.3 to {confidence_threshold:g})", value=len(possible)),
    gap="16px",
))
if api_key:
    print(f"Jev scored {len(jev.scores)} of {len(candidates) - len(exact)} candidates, cost ${jev.cost:.4f}.")
(
    matches.drop(columns="id")
    .rename(columns={"source": "Source", "title": "Title", "date": "Date", "p_same": "P(same)", "why": "Why"})
    .style.format({"P(same)": "{:.0%}"})
    .background_gradient(subset=["P(same)"], cmap="Greens", vmin=0, vmax=1)
    .set_properties(**{"text-align": "left"})
    .set_properties(subset=["Date"], **{"white-space": "nowrap"})
    .set_table_styles([{"selector": "th", "props": "text-align: left;"}])
    .hide(axis="index")
    if len(matches) else print("No other source has a record in this window.")
)


## 3. One event, every source

One column per source. **Orange cells** are rows where the sources disagree: the largest figure is at least 1.5 times the smallest. A figure with an asterisk (\*) is *modelled*: an estimate by the source (USGS PAGER), not a report.


In [ ]:
by_id = {c.id: c for c in candidates}
events = [seed] + [by_id[i] for i in joined.id]
details = M.fetch_details(api_url, token, events)
facts = M.fact_sheet(events, details.rows)
M.display_table(facts)


## 4. The spread, by source

The same numbers as bars: how far apart the sources are on deaths, people affected and people displaced.


In [ ]:
def metric_chart(label, title):
    columns = [c for c in facts.columns[:-2] if label in facts.index and pd.notna(facts.loc[label, c])]
    if not columns:
        return None
    chart = Chart(chart_type="bar", title=title, width=700, height=300)
    chart.set_options(scales={"x": {"type": "category"}, "y": {"beginAtZero": True}})
    for col in columns:
        name = col + (" (modelled)" if (label, col) in facts.attrs["modelled"] else "")
        chart.add_series(x=[label], y=[float(facts.loc[label, col])], name=name, series_type="bar",
                         color=M.COLOR_BY_LABEL.get(col))
    return chart


charts = [c for c in (metric_chart("Deaths", "Deaths"), metric_chart("People affected", "People affected"),
                      metric_chart("People displaced", "People displaced")) if c]
chart_view = Column(*charts, gap="12px") if charts else None
if chart_view is None:
    print("No joined source reported deaths, people affected or people displaced.")
chart_view


## 5. Where the records are

Every candidate, coloured by source. **Drag the slider** to change the minimum confidence: records join or drop out of the map as it moves (it works in the published page too). The selected event has a larger marker. Footprints are precise areas from the sources (Copernicus EMS areas, shake areas), never country outlines.


In [ ]:
import geopandas as gpd
import numpy as np
from lonboard import Map, PolygonLayer, ScatterplotLayer
from lonboard.layer_extension import DataFilterExtension
from manywidgets.lonboard import FilterBinder, LayerToggle


def rgb(hex_color, alpha=230):
    h = hex_color.lstrip("#")
    return [int(h[i : i + 2], 16) for i in (0, 2, 4)] + [alpha]


located = [(seed, 1.0, "selected event")] + [
    (by_id[r.id], r.p_same, r.why) for r in matches.itertuples() if by_id[r.id].point
]
labels = [M.SOURCE_LABELS.get(rec.source, rec.source) for rec, _, _ in located]
points_gdf = gpd.GeoDataFrame(
    {"title": [rec.title for rec, _, _ in located], "source": labels,
     "P(same)": [round(p, 2) for _, p, _ in located], "why": [why for _, _, why in located]},
    geometry=gpd.points_from_xy([rec.point[0] for rec, _, _ in located], [rec.point[1] for rec, _, _ in located]),
    crs="EPSG:4326",
)
is_seed = np.arange(len(located)) == 0
points = ScatterplotLayer.from_geopandas(
    points_gdf,
    get_fill_color=np.array([rgb(M.COLOR_BY_LABEL.get(label, "#777777")) for label in labels], dtype=np.uint8),
    get_line_color=np.where(is_seed[:, None], [0, 0, 0, 255], [255, 255, 255, 220]).astype(np.uint8),
    get_radius=np.where(is_seed, 12, 7),
    radius_units="pixels", stroked=True, line_width_min_pixels=2,
    pickable=True, auto_highlight=True,
    extensions=[DataFilterExtension(filter_size=1)],
    get_filter_value=points_gdf["P(same)"].to_numpy(dtype="float32"),
    filter_range=(confidence_threshold, 1.0),
)

shapes = M.footprints(events, details)
layers, controls = [], []
if shapes:
    shape_gdf = gpd.GeoDataFrame.from_features(shapes, crs="EPSG:4326")[["source", "title", "geometry"]]
    footprint_layer = PolygonLayer.from_geopandas(
        shape_gdf, get_fill_color=[213, 94, 0, 45], get_line_color=[213, 94, 0, 230],
        line_width_min_pixels=1, stroked=True, pickable=True,
    )
    layers.append(footprint_layer)
    controls.append(LayerToggle(layer=footprint_layer, label="Precise footprints"))
layers.append(points)

# open on the records that joined, not on every candidate (the filter hides the rest)
view = M.map_view([(rec.point[0], rec.point[1]) for rec, p, _ in located if p >= confidence_threshold])

slider = RangeSlider(min=0, max=1, low=confidence_threshold, high=1.0, step=0.05,
                     label="Minimum confidence: probability that the record is the same disaster")
binder = FilterBinder(slider, points)
legend = Legend(entries=[[M.COLOR_BY_LABEL.get(s, "#777777"), s] for s in dict.fromkeys(labels)], title="Source")
Column(Row(legend, *controls, gap="24px"), slider, binder, Map(layers, show_tooltip=True, height=520, view_state=view), gap="12px")


## 6. Outputs

`event.geojson` (the selected event, the records that joined it and the footprints) is written next to this page. The tables above are the results; nothing else is saved.

In [ ]:
out = Path(output_dir)
matched = [(seed, 1.0)] + [(by_id[r.id], r.p_same) for r in joined.itertuples()]
(out / "event.geojson").write_text(json.dumps(M.event_geojson(matched, shapes)))
print(f"Wrote event.geojson ({len(matched)} events, {len(shapes)} footprints) to {out}")

## Read this before trusting it

- **The scores are AI estimates.** Jev reads the records' titles, dates, places and magnitudes and says how likely they describe one disaster. It can be wrong, in both directions. The default threshold is 0.6; move it (the slider above, or the run form) and see what changes.
- **The search starts from the selected record.** A record linked to the disaster only through another record, or outside the time window and radius, is not found. Widen the windows in the run form if you suspect that. Other records of the selected event's own source are not candidates, unless Montandon links them to it.
- **Exact links are rare for now.** A record that Montandon itself links to the selected event joins without asking Jev, but the catalogue holds few such links so far (CEMS to GDACS and Charter events), so most joins here come from Jev. The notebook never infers a link from an id or a register number.
- **Numbers that disagree are not errors in this notebook.** Sources measure different things (reported against modelled, deaths against people killed or missing, a country against one area of interest) and the highlighted rows show exactly that. A figure with an asterisk is *modelled*: USGS's estimate. Copernicus figures are summed over its areas of interest, a register's records for different countries (EM-DAT, GLIDE) add up, and otherwise the largest figure is shown. EM-DAT publishes damage in thousands of US dollars, but its unit in the catalogue says only "count": amounts in different units are never compared.
- **Candidates are capped** to the most plausible by time and distance (a parameter), so a very busy window can hide a record.
